# Notebook 2 — KPIs & Visuals

**Purpose:** Compute all core game economy KPIs from the raw data — replicating the logic in the `curated.*` SQL views — then produce polished charts saved to `reports/figures/`.

**KPIs covered:** DAU, Revenue, ARPU, ARPPU, Payer Conversion, D1/D7/D30 Retention, Conversion Funnel.

## Cell 1 — Imports & DB Connection

Load libraries, configure seaborn style, and open a connection to PostgreSQL using credentials from `.env`. All KPI data comes from the `curated.*` views — no local CSV files are read.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()  # reads .env from cwd or any parent directory

sns.set_theme(style="whitegrid")

db_url = os.getenv("DATABASE_URL")
if not db_url:
    raise EnvironmentError(
        "DATABASE_URL is not set. "
        "Copy .env.example to .env and fill in your credentials:\n"
        "  cp .env.example .env"
    )

engine = create_engine(db_url)

FIGURES_DIR = Path("..") / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Connected to:", engine.url.render_as_string(hide_password=True))
print("Figures will be saved to:", FIGURES_DIR.resolve())

## Cell 2 — Load Curated Views from PostgreSQL

Query each `curated.*` view directly from the database. These views encapsulate all KPI logic in SQL (see `sql/02_kpi_queries.sql`) — the notebook is purely for visualisation and summary.

In [ ]:
dau       = pd.read_sql("SELECT * FROM curated.daily_dau",        engine, parse_dates=True)
revenue   = pd.read_sql("SELECT * FROM curated.daily_revenue",     engine, parse_dates=True)
arpu      = pd.read_sql("SELECT * FROM curated.daily_arpu_arppu",  engine, parse_dates=True)
retention = pd.read_sql("SELECT * FROM curated.retention_cohorts", engine, parse_dates=True)
funnel    = pd.read_sql("SELECT * FROM curated.funnel",            engine, parse_dates=True)

# Ensure date columns are pandas datetime regardless of DB driver behaviour
dau["activity_date"]        = pd.to_datetime(dau["activity_date"])
arpu["activity_date"]       = pd.to_datetime(arpu["activity_date"])
revenue["revenue_date"]     = pd.to_datetime(revenue["revenue_date"])
retention["cohort_date"]    = pd.to_datetime(retention["cohort_date"])

# Derived columns used in charts
arpu["payer_conversion"] = arpu["payers"] / arpu["dau"]
funnel["pct_of_installed"] = funnel["users"] / funnel["users"].iloc[0]
funnel["drop_off_pct"] = 1 - funnel["users"] / funnel["users"].shift(1).fillna(funnel["users"].iloc[0])

print("Curated views loaded.")
display(arpu.head())

## Cell 3 — DAU Over Time

Daily Active Users (DAU) is the foundation of all engagement and monetisation benchmarking. A stable or growing DAU is a prerequisite for healthy ARPU trends.

In [ ]:
dau_sorted = dau.sort_values("activity_date")

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(dau_sorted["activity_date"], dau_sorted["dau"],
        color="steelblue", linewidth=1.8, label="DAU")
ax.plot(dau_sorted["activity_date"], dau_sorted["dau"].rolling(7).mean(),
        color="darkorange", linewidth=2, linestyle="--", label="7-day rolling avg")

ax.set_title("Daily Active Users (DAU) Over Time", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("DAU")
ax.legend()
fig.autofmt_xdate()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "dau_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'dau_over_time.png'}")

## Cell 4 — Revenue, ARPU & ARPPU Over Time

**ARPU** (Average Revenue Per User) measures monetisation efficiency across the full DAU base. **ARPPU** (Average Revenue Per Paying User) isolates payer spend behaviour. Divergence between the two signals changes in payer conversion rate.

In [ ]:
arpu_plot = arpu.sort_values("activity_date")

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(arpu_plot["activity_date"], arpu_plot["total_revenue"],
        color="steelblue", alpha=0.45, label="Total Revenue")
ax1.set_xlabel("Date")
ax1.set_ylabel("Revenue (USD)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.plot(arpu_plot["activity_date"], arpu_plot["arpu"],
         color="darkorange", linewidth=2, label="ARPU")
ax2.plot(arpu_plot["activity_date"], arpu_plot["arppu"],
         color="crimson", linewidth=2, linestyle="--", label="ARPPU")
ax2.set_ylabel("Per-User Revenue (USD)", color="black")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Daily Revenue, ARPU & ARPPU", fontsize=14, fontweight="bold")
fig.autofmt_xdate()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "arpu_arppu_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'arpu_arppu_trend.png'}")

## Cell 5 — Payer Conversion Rate Over Time

Payer conversion is the percentage of daily active users who made at least one IAP purchase that day. A rising conversion rate means more of the active base is monetising, which is typically the highest-leverage lever in free-to-play economies.

In [ ]:
overall_conversion = arpu["payers"].sum() / arpu["dau"].sum()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(arpu_plot["activity_date"], arpu_plot["payer_conversion"],
        color="mediumseagreen", linewidth=1.8)
ax.fill_between(arpu_plot["activity_date"], arpu_plot["payer_conversion"],
                alpha=0.2, color="mediumseagreen")
ax.axhline(overall_conversion, color="red", linestyle="--", linewidth=1.2,
           label=f"Overall avg: {overall_conversion:.1%}")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=1))

ax.set_title("Daily Payer Conversion Rate", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Payer Conversion Rate")
ax.legend()
fig.autofmt_xdate()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "payer_conversion.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'payer_conversion.png'}")

## Cell 6 — Retention Cohort Heatmap

Cohort retention tracks what percentage of users from each install date were active on Day 1, Day 7, and Day 30. This is the single most important metric for predicting long-term LTV and diagnosing early drop-off.

In [ ]:
ret_matrix = retention.set_index("cohort_date")[["d1_retention_rate", "d7_retention_rate", "d30_retention_rate"]]
ret_matrix.columns = ["D1", "D7", "D30"]

# Cohorts from the last 30 days won't have observed D30 data yet — exclude to avoid misleading zeros
cutoff = pd.to_datetime(ret_matrix.index.max()) - pd.Timedelta(days=30)
ret_matrix_plot = ret_matrix[pd.to_datetime(ret_matrix.index) <= cutoff]
if ret_matrix_plot.empty:
    ret_matrix_plot = ret_matrix

fig, ax = plt.subplots(figsize=(8, max(6, len(ret_matrix_plot) * 0.35)))

sns.heatmap(
    ret_matrix_plot,
    annot=True, fmt=".0%",
    cmap="YlOrRd",
    linewidths=0.4,
    ax=ax,
    vmin=0, vmax=1,
    cbar_kws={"label": "Retention Rate"}
)

ax.set_title("D1 / D7 / D30 Retention by Install Cohort", fontsize=14, fontweight="bold")
ax.set_xlabel("Retention Day")
ax.set_ylabel("Install Date (Cohort)")
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "retention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'retention_heatmap.png'}")

## Cell 7 — Conversion Funnel

The funnel tracks user progression through four stages: install → session → soft currency spend → real-money purchase. Each stage's drop-off reveals where the biggest optimisation opportunities lie.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

palette = sns.color_palette("Blues_d", len(funnel))
bars = ax.barh(
    funnel["step"][::-1].values,
    funnel["users"][::-1].values,
    color=palette
)

funnel_rev = funnel[::-1].reset_index(drop=True)
for i, bar in enumerate(bars):
    row = funnel_rev.iloc[i]
    # User count label at end of bar
    ax.text(
        bar.get_width() * 1.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['users']):,}",
        va="center", ha="left", fontsize=10
    )
    # Drop-off annotation inside bar
    if row["drop_off_pct"] > 0:
        ax.text(
            bar.get_width() * 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"-{row['drop_off_pct']:.0%} drop",
            va="center", ha="center",
            fontsize=9, color="white", fontweight="bold"
        )

ax.set_title("User Conversion Funnel", fontsize=14, fontweight="bold")
ax.set_xlabel("Users")
ax.set_ylabel("Funnel Stage")
ax.set_xlim(0, funnel["users"].max() * 1.18)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "conversion_funnel.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'conversion_funnel.png'}")

print("\nFunnel summary:")
display(funnel[["step", "users", "pct_of_installed", "drop_off_pct"]].round(3))

## Cell 8 — Key Metrics Summary

Single-number summary of all core KPIs for the executive report and portfolio write-up.

In [ ]:
avg_dau       = arpu["dau"].mean()
total_rev     = arpu["total_revenue"].sum()
overall_arpu  = total_rev / arpu["dau"].sum()
overall_arppu = total_rev / arpu["payers"].sum()
payer_conv    = arpu["payers"].sum() / arpu["dau"].sum()

avg_d1  = retention["d1_retention_rate"].mean()
avg_d7  = retention["d7_retention_rate"].mean()
avg_d30 = retention["d30_retention_rate"].mean()

summary = pd.DataFrame([
    {"Metric": "Average DAU",           "Value": f"{avg_dau:,.0f}"},
    {"Metric": "Total Revenue",         "Value": f"${total_rev:,.2f}"},
    {"Metric": "Overall ARPU",          "Value": f"${overall_arpu:.4f}"},
    {"Metric": "Overall ARPPU",         "Value": f"${overall_arppu:.2f}"},
    {"Metric": "Payer Conversion Rate", "Value": f"{payer_conv:.1%}"},
    {"Metric": "Avg D1 Retention",      "Value": f"{avg_d1:.1%}"},
    {"Metric": "Avg D7 Retention",      "Value": f"{avg_d7:.1%}"},
    {"Metric": "Avg D30 Retention",     "Value": f"{avg_d30:.1%}"},
])

display(summary.style.hide(axis="index").set_caption("Game Economy — Key Metrics Summary"))